# A/B Test 04 — Average Revenue Per User (ARPU)

**Question.** Did the treatment increase revenue per user?

**Key modeling decision.** ARPU is revenue **per user**, measured across the **entire** user base — **no filter**. Non-buyers count and contribute 0. (Contrast with AOV, which filters to buyers.)

**Metric type.** `order_value` is continuous → compare **means** → **Welch's t-test**.


In [1]:
import pandas as pd
from scipy import stats

df = pd.read_parquet("../data/ab_test_data.parquet")
df.head()

,user_id,group,clicked,purchased,order_value,returned_30d
0,T01163,treatment,1,0,0.00,1
1,T04385,treatment,0,0,0.00,1
2,C01902,control,1,1,96.31,0
3,C03397,control,0,0,0.00,0
4,T05695,treatment,1,0,0.00,1


In [2]:
# Data quality checks (run before trusting any metric)
print("shape:", df.shape)
print("\nnulls:\n", df.isnull().sum())
print("\nduplicate user_id:", df["user_id"].duplicated().sum())

# SRM (sample ratio mismatch): groups should be ~50/50
print("\ngroup sizes:\n", df["group"].value_counts())

# Binary columns must contain only 0/1
for col in ["clicked", "purchased", "returned_30d"]:
    print(col, "unique:", sorted(df[col].unique()))

shape: (12000, 6)

nulls:
 user_id         0
group           0
clicked         0
purchased       0
order_value     0
returned_30d    0
dtype: int64

duplicate user_id: 0

group sizes:
 group
treatment    6000
control      6000
Name: count, dtype: int64
clicked unique: [np.int64(0), np.int64(1)]
purchased unique: [np.int64(0), np.int64(1)]
returned_30d unique: [np.int64(0), np.int64(1)]


## Compute ARPU — mean order_value over ALL users (no filter)

In [3]:
print(df.groupby("group")["order_value"].agg(["mean", "sum", "count"]))

               mean       sum  count
group                               
control    4.693533  28161.20   6000
treatment  7.900247  47401.48   6000


## Statistical test — Welch's t-test

In [4]:
a = df[df["group"] == "control"]["order_value"]
b = df[df["group"] == "treatment"]["order_value"]

stat, pval = stats.ttest_ind(a, b, equal_var=False)
print(f"control ARPU = {a.mean():.2f} | treatment ARPU = {b.mean():.2f}")
print(f"p-value = {pval:.4f}")
print("Significant" if pval < 0.05 else "Not significant")

control ARPU = 4.69 | treatment ARPU = 7.90
p-value = 0.0000
Significant


## Result

ARPU rose from **4.69 (control)** to **7.90 (treatment)**, p ≈ 0.0000 → significant (**+68%**).

Here `count = 6000` is the total users per group (buyers + non-buyers), not the number of buyers — most rows have `order_value = 0`.

## Concepts

### Where did the lift come from? (decompose the levers)
ARPU is the product of two levers:

> **ARPU = conversion rate × AOV**

| Lever | Control | Treatment | Change |
|---|---|---|---|
| Conversion rate | 0.0545 | 0.0918 | **+68%** |
| AOV | 86.12 | 86.03 | ~0% (flat) |
| **ARPU** | 4.69 | 7.90 | **+68%** |

Math check: 0.0545 × 86.12 ≈ 4.69 and 0.0918 × 86.03 ≈ 7.90.

Since AOV is flat, the entire ARPU increase comes from **conversion** — more buyers, not bigger orders. The ARPU lift mirrors the conversion lift exactly (both ≈ +68%).

### Is a t-test valid on skewed, zero-inflated data?
The raw distribution is highly skewed (thousands of zeros + a few hundred ≈ 86 values), but the t-test does not require the **raw data** to be normal — only the **sampling distribution of the mean**. By the **Central Limit Theorem**, with 6000 users per group the mean is well-behaved regardless of the skew, so the t-test is valid. With a small sample, a non-parametric test (**Mann-Whitney U**) or **bootstrapping** would be safer.
